# Ejercicio 5: Espacio Vectorial

## Objetivo de la práctica
- Implementar un Sistema de Recuperación de Información completo, desde la lectura del corpus hasta la recuperación de resultados.

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus
2. Realiza las etapas de preprocesamiento sobre el corpus


In [1]:
import pandas as pd
import re
from nltk.stem import PorterStemmer


df_corpus = pd.read_csv('wikipedia_text_corpus.csv', nrows=100)
print(f"Corpus cargado: {len(df_corpus)} documentos")


def preprocess_text(text):
    cleaned = re.sub(r'[^\w\s]', ' ', text)
    cleaned = re.sub(r'\d+', '', cleaned)
    cleaned = cleaned.lower()
    tokens = cleaned.split()
    stemmer = PorterStemmer()
    stemmed_tokens = [stemmer.stem(token) for token in tokens]

    return ' '.join(stemmed_tokens)


df_corpus['processed_text'] = df_corpus['text'].apply(preprocess_text)

print("Preprocesamiento completado.")
print(f"Ejemplo de documento original:\n{df_corpus['text'].iloc[0][:200]}...")
print(f"Ejemplo de documento procesado:\n{df_corpus['processed_text'].iloc[0][:200]}...")

Corpus cargado: 100 documentos
Preprocesamiento completado.
Ejemplo de documento original:
Anovo

Anovo (formerly A Novo) is a computer services company based in Beauvais, France. It was founded in 1987, went public in 1999, and is currently a member of the CAC Small.

It won in the categor...
Ejemplo de documento procesado:
anovo anovo formerli a novo is a comput servic compani base in beauvai franc it wa found in went public in and is current a member of the cac small it won in the categori servic and repair of the mobi...


## Parte 1: Recuperación con TF-IDF

### Actividad:
3. Obtén la representación vectorial de los documentos utilizando el modelo TF-IDF
4. A partir de un conjunto de 10 queries, verifica la recuperación del sistema

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df_corpus['processed_text'])

print(f"Matriz TF-IDF: {tfidf_matrix.shape[0]} documentos x {tfidf_matrix.shape[1]} términos")


queries = [
    "U.S. Army Regulation",
    "conductive polymers",
    "global warming",
    "world war two battles",
    "chemical compounds",
    "hybrid vehicles",
    "cryptocurrency trends",
    "algebra equations",
    "democracy government",
    "art painting"
]

def search_tfidf(query, vectorizer, tfidf_matrix, df, top_n=5):
    
    query_processed = preprocess_text(query)
    query_vec = vectorizer.transform([query_processed])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    return top_indices, scores

print("\n" + "="*60)
print("RECUPERACIÓN CON TF-IDF")
print("="*60)

for query in queries:
    top_idx, scores = search_tfidf(query, vectorizer, tfidf_matrix, df_corpus)
    print(f"\nQuery: '{query}'")
    for rank, idx in enumerate(top_idx, 1):
        title = df_corpus['title'].iloc[idx] if 'title' in df_corpus.columns else f"Doc {idx}"
        print(f"  {rank}. [{scores[idx]:.4f}] {title}")

Matriz TF-IDF: 100 documentos x 6630 términos

RECUPERACIÓN CON TF-IDF

Query: 'U.S. Army Regulation'
  1. [0.1050] Doc 44
  2. [0.1023] Doc 7
  3. [0.0643] Doc 64
  4. [0.0614] Doc 67
  5. [0.0459] Doc 93

Query: 'conductive polymers'
  1. [0.6650] Doc 9
  2. [0.1702] Doc 22
  3. [0.0462] Doc 34
  4. [0.0430] Doc 49
  5. [0.0358] Doc 44

Query: 'global warming'
  1. [0.0676] Doc 11
  2. [0.0528] Doc 78
  3. [0.0303] Doc 73
  4. [0.0288] Doc 58
  5. [0.0286] Doc 6

Query: 'world war two battles'
  1. [0.1021] Doc 61
  2. [0.0479] Doc 64
  3. [0.0471] Doc 86
  4. [0.0454] Doc 34
  5. [0.0422] Doc 39

Query: 'chemical compounds'
  1. [0.1899] Doc 7
  2. [0.0790] Doc 9
  3. [0.0408] Doc 21
  4. [0.0220] Doc 44
  5. [0.0196] Doc 32

Query: 'hybrid vehicles'
  1. [0.4029] Doc 86
  2. [0.0798] Doc 88
  3. [0.0744] Doc 7
  4. [0.0665] Doc 73
  5. [0.0276] Doc 37

Query: 'cryptocurrency trends'
  1. [0.0789] Doc 10
  2. [0.0584] Doc 59
  3. [0.0491] Doc 34
  4. [0.0000] Doc 99
  5. [0.0000] Do

## Parte 2: Recuperación con BM25

### Actividad:
5. Implementa un sistema de recuperación usando el modelo BM25.
6. Para el mismo conjunto de 10 queries, verifica la recuperación del sistema

In [3]:
from rank_bm25 import BM25Okapi


tokenized_corpus = [doc.split() for doc in df_corpus['processed_text']]

bm25 = BM25Okapi(tokenized_corpus)

print(f"Modelo BM25 creado con {len(tokenized_corpus)} documentos")

def search_bm25(query, bm25, df, top_n=5):
    
    query_processed = preprocess_text(query)
    query_tokens = query_processed.split()
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:top_n]
    return top_indices, scores

print("\n" + "="*60)
print("RECUPERACIÓN CON BM25")
print("="*60)

for query in queries:
    top_idx, scores = search_bm25(query, bm25, df_corpus)
    print(f"\nQuery: '{query}'")
    for rank, idx in enumerate(top_idx, 1):
        title = df_corpus['title'].iloc[idx] if 'title' in df_corpus.columns else f"Doc {idx}"
        print(f"  {rank}. [{scores[idx]:.4f}] {title}")

Modelo BM25 creado con 100 documentos

RECUPERACIÓN CON BM25

Query: 'U.S. Army Regulation'
  1. [13.1507] Doc 7
  2. [9.0003] Doc 44
  3. [7.8453] Doc 93
  4. [7.1879] Doc 10
  5. [6.3390] Doc 64

Query: 'conductive polymers'
  1. [10.4592] Doc 9
  2. [5.6467] Doc 22
  3. [3.8451] Doc 34
  4. [3.5903] Doc 91
  5. [3.4249] Doc 85

Query: 'global warming'
  1. [7.1550] Doc 78
  2. [5.7509] Doc 6
  3. [4.8895] Doc 83
  4. [4.8347] Doc 11
  5. [4.3017] Doc 73

Query: 'world war two battles'
  1. [12.1945] Doc 61
  2. [6.4096] Doc 64
  3. [5.5104] Doc 33
  4. [4.5098] Doc 14
  5. [3.6106] Doc 86

Query: 'chemical compounds'
  1. [8.7243] Doc 7
  2. [8.0007] Doc 9
  3. [3.7671] Doc 21
  4. [3.1455] Doc 32
  5. [2.8720] Doc 44

Query: 'hybrid vehicles'
  1. [9.7018] Doc 86
  2. [3.9310] Doc 88
  3. [3.7352] Doc 73
  4. [3.6986] Doc 7
  5. [3.1083] Doc 9

Query: 'cryptocurrency trends'
  1. [7.4071] Doc 10
  2. [5.8634] Doc 59
  3. [5.2772] Doc 34
  4. [0.0000] Doc 99
  5. [0.0000] Doc 95

Qu

## Parte 3: Comparación de resultados

### Actividad:
7. Verifica cuáles documentos son recuperados (y en qué orden) por cada modelo de recuperación 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

TOP_N = 5


print("TABLA DETALLADA: TOP-5 POR QUERY")

detail_rows = []
summary_rows = []

for query in queries:
    ti_idx, ti_scores = search_tfidf(query, vectorizer, tfidf_matrix, df_corpus, top_n=TOP_N)
    bm_idx, bm_scores = search_bm25(query, bm25, df_corpus, top_n=TOP_N)

    for rank in range(TOP_N):
        detail_rows.append({
            'Query':          query,
            'Rank':           rank + 1,
            'TF-IDF Doc':    f"Doc {ti_idx[rank]}",
            'TF-IDF Score':   round(ti_scores[ti_idx[rank]], 4),
            'BM25 Doc':      f"Doc {bm_idx[rank]}",
            'BM25 Score':     round(bm_scores[bm_idx[rank]], 4),
            'Mismo doc':     'Si' if ti_idx[rank] == bm_idx[rank] else 'No',
        })

    overlap  = len(set(ti_idx) & set(bm_idx))
    acuerdan = int(ti_idx[0]) == int(bm_idx[0])
    summary_rows.append({
        'Query':            query,
        'TF-IDF #1':       f"Doc {ti_idx[0]}",
        'BM25 #1':         f"Doc {bm_idx[0]}",
        'Coinciden top-5': overlap,
        'Mismo doc #1':    'Si' if acuerdan else 'No',
    })

df_detail  = pd.DataFrame(detail_rows)
df_summary = pd.DataFrame(summary_rows)

# Imprimir tabla detallada query por query
for query in queries:
    print("\n" + "=" * 65)
    print(f"Query: '{query}'")
    print("=" * 65)
    sub = df_detail[df_detail['Query'] == query][['Rank', 'TF-IDF Doc', 'TF-IDF Score', 'BM25 Doc', 'BM25 Score', 'Mismo doc']]
    print(sub.to_string(index=False))

# ── Tabla resumen ─────────────────────────────────────────────────────────
print("\n\n" + "=" * 75)
print("RESUMEN: TF-IDF vs BM25 (doc #1 y coincidencias en top-5)")
print("=" * 75)
print(df_summary.to_string(index=False))

# ── Graficas ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
labels = [f"Q{i+1}" for i in range(len(queries))]

# Grafica 1: docs coincidentes en top-5
axes[0].bar(labels, df_summary['Coinciden top-5'], color='steelblue', edgecolor='white', zorder=3)
axes[0].axhline(TOP_N, color='red', linestyle='--', linewidth=1, label=f'Maximo ({TOP_N})')
axes[0].set_title('Documentos coincidentes en Top-5', fontsize=12)
axes[0].set_xlabel('Query')
axes[0].set_ylabel('Num de docs en comun')
axes[0].set_ylim(0, TOP_N + 1)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3, zorder=0)

# Grafica 2: acuerdan en doc #1?
acuerdo_vals = [1 if v == 'Si' else 0 for v in df_summary['Mismo doc #1']]
colors = ['#2ecc71' if v else '#e74c3c' for v in acuerdo_vals]
axes[1].bar(labels, acuerdo_vals, color=colors, edgecolor='white', zorder=3)
axes[1].set_title('Ambos modelos coinciden en el Doc #1?', fontsize=12)
axes[1].set_xlabel('Query')
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['No', 'Si'])
axes[1].grid(axis='y', alpha=0.3, zorder=0)
axes[1].legend(handles=[mpatches.Patch(color='#2ecc71', label='Si'),
                         mpatches.Patch(color='#e74c3c', label='No')])

plt.suptitle('Comparacion TF-IDF vs BM25', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nLeyenda:")
for i, q in enumerate(queries):
    print(f"  Q{i+1}: {q}")

TypeError: can only concatenate str (not "int") to str